# Baseline-anchored context correction: development screen

This notebook tests a bounded Long-Context correction to a freshly trained copy of the authors' exact dataset-specific model architecture. It uses only the published training cohort and saved protein-grouped out-of-fold context predictions from the completed hCKSAAP or UbiComb benchmark. The released authors' checkpoint provides the architecture; its in-sample predictions are **not** used to fit the correction.

This is a post-historical-test, **development-only** screen. It never loads the released independent test cohort. Any later evaluation on the already-viewed hCKSAAP/UbiComb tests must be labelled exploratory. Select **GPU T4 ×2**, enable Internet, and attach the completed prior Kaggle notebook's **Output** as an input dataset. Add a read-only `GITHUB_TOKEN` secret for the private repository.


## Repository and dependencies
The printed commit identifies the implementation used by this run. For a resumed run, set `REPO_REF` to that exact commit before execution.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys, threading, time
from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

GITHUB_TOKEN = secret('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Add read-only GITHUB_TOKEN in Kaggle Secrets.')
REPO_ROOT = Path('/kaggle/working/Ubiquitination-Sites-Prediction')
REPO_REF = None  # Use the original printed commit when resuming.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN'] = GITHUB_TOKEN
askpass = Path('/kaggle/working/ubipred_anchored_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
askpass.chmod(0o700)
git_env['GIT_ASKPASS'] = str(askpass)
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules',
                        'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
                        str(REPO_ROOT)], check=True, env=git_env)
    elif REPO_REF is None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
    if REPO_REF is not None:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', REPO_REF], check=True, env=git_env)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
                   check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
%pip install -q -r experiment/requirements-colab.txt h5py gdown
tf_version = importlib.metadata.version('tensorflow')
legacy_requirement = 'tf-keras==' + '.'.join(tf_version.split('.')[:2]) + '.*'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', legacy_requirement], check=True)
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2 before training.')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])


## Prior run and model artifact
Select `hcksaap` or `ubicomb`. Attach the completed corresponding notebook-23 or notebook-24 Output in Kaggle's **Add Input** panel. The notebook locates the development summary, training cohort, fold assignments and context predictions under that input. The authors' public checkpoint is downloaded only if it is not already attached.

In [ ]:
import gdown

BENCHMARK = 'ubicomb'  # Change to 'hcksaap' for the human benchmark.
MODEL_INFO = {
    'hcksaap': ('DeepUBImodel_HubiPred_AAindex_One_Hot_Emb_drop_out_changed73887.h5',
                '1v6u6N6JpqJs8bi_3wLhp5gcbS7uxsAwD',
                'fef6859092626f94e6a7a70e644c37f2f68bf93ec00f251c2003458dccb90526'),
    'ubicomb': ('UBICOMB_DeepUBI_AAindex_One_Hot_ProtT5_drop_out_Changed3257.h5',
                '1jGURx76ll40d0TebNO9CQOcG9SF8V0xT',
                'a1057c0e13d6dbd1348f83b26ed0fa347498dec914b7c671f274713fdc6cdefd'),
}
if BENCHMARK not in MODEL_INFO:
    raise ValueError('BENCHMARK must be hcksaap or ubicomb.')
PRIOR_RUN_DIR = None  # Set explicitly only if discovery finds multiple runs.
if PRIOR_RUN_DIR is None:
    matches = []
    for root in (Path('/kaggle/input'), Path('/kaggle/working/mmubipred_benchmark_runs')):
        if root.exists():
            for summary_path in root.rglob('development_summary.json'):
                candidate = summary_path.parent
                if (candidate / 'train_cohort.json').is_file() and \
                   json.loads(summary_path.read_text()).get('benchmark') == BENCHMARK:
                    matches.append(candidate)
    if len(matches) != 1:
        raise RuntimeError(f'Expected one attached {BENCHMARK} run; found {matches}. Attach its Output or set PRIOR_RUN_DIR.')
    PRIOR_RUN_DIR = matches[0]
PRIOR_RUN_DIR = Path(PRIOR_RUN_DIR)
MODEL_NAME, MODEL_ID, MODEL_SHA256 = MODEL_INFO[BENCHMARK]
MODEL_DIR = Path('/kaggle/working/anchored_context_source') / BENCHMARK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = MODEL_DIR / MODEL_NAME
if not MODEL_FILE.is_file():
    attached = list(Path('/kaggle/input').rglob(MODEL_NAME))
    if len(attached) > 1:
        raise RuntimeError(f'Multiple attached copies of {MODEL_NAME}: {attached}')
    if attached:
        shutil.copy2(attached[0], MODEL_FILE)
    else:
        gdown.download(id=MODEL_ID, output=str(MODEL_FILE), quiet=False)
def sha256(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()
if sha256(MODEL_FILE) != MODEL_SHA256:
    raise RuntimeError('Authors checkpoint checksum differs from the verified artifact.')
RUN_DIR = Path('/kaggle/working/anchored_context_screen_v1') / BENCHMARK
RESUME_RUN_DIR = None  # Attach a prior screen Output and set its run directory here to resume.
if RESUME_RUN_DIR is not None:
    previous = Path(RESUME_RUN_DIR)
    if not (previous / 'contract.json').is_file():
        raise FileNotFoundError(f'Missing screen contract: {previous}')
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise RuntimeError(f'Current run directory is not empty: {RUN_DIR}')
    shutil.copytree(previous, RUN_DIR, dirs_exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = Path('experiment/configs/anchored_context_correction_screen_v1.json')
AAINDEX = Path('replication/MMUbiPred/aaindex31.txt')
print('Prior development run:', PRIOR_RUN_DIR)
print('New screen run:', RUN_DIR)
print('Verified authors checkpoint:', MODEL_FILE)


## Preflight
This verifies prior artifact hashes and checks that the exact released architecture can be cloned with fresh weights. It does not load the published test cohort.

In [ ]:
def command(stage_name, *extra):
    return [sys.executable, '-u', 'experiment/scripts/run_anchored_context_screen.py',
            stage_name, '--config', str(CONFIG), '--prior-run-dir', str(PRIOR_RUN_DIR),
            '--released-model', str(MODEL_FILE), '--aaindex', str(AAINDEX),
            '--output-dir', str(RUN_DIR), *map(str, extra)]
def run_live(parts, env=None):
    print('$', ' '.join(parts), flush=True)
    process = subprocess.Popen(parts, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, parts)
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', 'experiment/tests',
          '-p', 'test_anchored*.py'])
run_live(command('preflight'))


## Five protein-grouped exact-anchor folds
Each fold trains three seeds of a fresh copy of the released model architecture for all 15 epochs, selects a checkpoint by inner validation, and predicts only its held-out proteins. GPU 0 processes folds 0, 2 and 4; GPU 1 processes folds 1 and 3. Completed folds are skipped and an interrupted candidate resumes after its last fully saved epoch.

In [ ]:
def run_parallel_folds():
    workers, readers = [], []
    def stream(process, gpu):
        for line in process.stdout:
            print(f'[T4 {gpu}] {line}', end='', flush=True)
    try:
        for gpu, folds in ((0, [0, 2, 4]), (1, [1, 3])):
            env = os.environ.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
            env['TF_USE_LEGACY_KERAS'] = '1'
            env['OMP_NUM_THREADS'] = '2'
            parts = command('fold', '--fold', folds[0])
            # One process per GPU executes its assigned folds sequentially.
            def worker_command(fold):
                return command('fold', '--fold', fold)
            for fold in folds:
                if (RUN_DIR / 'folds' / f'fold_{fold}' / 'selection.json').is_file():
                    print(f'Fold {fold} already complete; subprocess will verify and skip.')
            script = 'import subprocess,sys\n' + '\n'.join(
                'subprocess.run(' + repr(worker_command(fold)) + ',check=True)'
                for fold in folds)
            print(f'Starting T4 {gpu} on folds {folds}', flush=True)
            process = subprocess.Popen([sys.executable, '-u', '-c', script], env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1)
            workers.append((gpu, process))
            reader = threading.Thread(target=stream, args=(process, gpu), daemon=True)
            reader.start()
            readers.append(reader)
        while True:
            failures = [(gpu, process.returncode) for gpu, process in workers
                        if process.poll() is not None and process.returncode != 0]
            if failures:
                raise RuntimeError(f'Fold worker failed: {failures}')
            if all(process.poll() == 0 for _, process in workers):
                break
            time.sleep(1)
    except BaseException:
        for _, process in workers:
            if process.poll() is None:
                process.terminate()
        for _, process in workers:
            process.wait()
        raise
    finally:
        for reader in readers:
            reader.join(timeout=5)
run_parallel_folds()


## Development-only decision
The bounded correction is cross-fitted from other folds' anchor and context predictions. It is compared with the untouched exact-architecture anchor and an intercept-only adjustment. The fixed decision rule requires material MCC gain, gain beyond intercept-only calibration, consistency across folds, and bounded losses on other metrics. No released-test inference is performed.

In [ ]:
run_live(command('analyze'))
screen = json.loads((RUN_DIR / 'development_screen.json').read_text())
display({key: screen[key] for key in ('status', 'checks', 'released_independent_test_accessed')})
display(screen['metrics'])
display(screen['by_fold'])


## Persistence and interpretation
The output directory is `/kaggle/working/anchored_context_screen_v1/<benchmark>`. Save the Kaggle version or export this entire directory before switching accounts. To resume elsewhere, attach the saved Output and set `RESUME_RUN_DIR` to the directory containing `contract.json`; keep the original repository commit. Training restarts at the last completed epoch if an epoch was interrupted. The saved model/optimizer state is restored, but exact bitwise replay after a process interruption is not guaranteed across runtime images.

A `STOP_NO_MATERIAL_DEVELOPMENT_GAIN` result means no full-data refit or released-test run is justified by this screen. An `ADVANCE_TO_REFIT_PLANNING` result authorizes planning a separate refit; it is **not** an independent superiority result. The hCKSAAP and UbiComb test sets have already been inspected in earlier work.